In [1]:
!pip install pandas matplotlib seaborn wordcloud

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Read Fana comments
df = pd.read_json('fana_comments.json')

# Normalize column name to 'comment' if needed
if 0 in df.columns:
    df.rename(columns={0: 'comment'}, inplace=True)
elif '0' in df.columns:
    df.rename(columns={'0': 'comment'}, inplace=True)
elif df.columns[0] != 'comment':
    df.rename(columns={df.columns[0]: 'comment'}, inplace=True)

df.head()

,comment
0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው
1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...
2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏
3,😢😢😢😢😭😭😭
4,😢😢😢😢😢


In [4]:
df.tail()

,comment
55355,الحمدلله رب العالمين 👍👍👍👍🇪🇹
55356,والله مو فاهم شيء 😁
55357,በአረቡ አለም መግኘት የነበረብንን ጎረቤት እንደመሆናችን ብዙ ነ...
55358,ዋዉ በጣም አሪፍ ነዉ
55359,በጣማ ነው ደስ የሚለው


In [5]:
df.shape

(55360, 1)

In [6]:
# Make sure column is 'comment' in case we missed it
if 'comment' not in df.columns and len(df.columns) == 1:
    df.columns = ['comment']
df = df.reset_index()
df.head()

,index,comment
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏
3,3,😢😢😢😢😭😭😭
4,4,😢😢😢😢😢


In [7]:
duplicate_count = df.duplicated().sum()
print(f"Total duplicates found: {duplicate_count}")

Total duplicates found: 0


In [8]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
index      0
comment    0
dtype: int64


In [9]:
import re

def extract_urls(text):
    return re.findall(r'https?://\S+|www\.\S+', str(text))

# Apply to dataframe
df['urls'] = df['comment'].apply(extract_urls)

# Show rows with URLs
df[df['urls'].str.len() > 0][['comment', 'urls']].head()

,comment,urls
66,https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6TaF,[https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6...
2136,https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrkF0_,[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...
2137,"( Lazy, Bankrupt, coward, Selfish) https://you...",[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...
2335,https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjcOzY,[https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjc...
3133,https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrSbZQ,[https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrS...


In [10]:
import re

# Function to remove URLs
def remove_urls(text):
    return re.sub(r'https?://[^\s]+|www\.[^\s]+', '', str(text))

# Apply cleaning
df['clean_comment'] = df['comment'].apply(remove_urls)

# Show rows where URLs existed (compare before vs after)
df[df['urls'].str.len() > 0][['comment', 'urls', 'clean_comment']].head(5)

,comment,urls,clean_comment
66,https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6TaF,[https://youtu.be/Iduy3flYlF0?si=DkHgJg2JkP-Q6...,
2136,https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrkF0_,[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...,
2137,"( Lazy, Bankrupt, coward, Selfish) https://you...",[https://youtu.be/8HIRHYP3HAY?si=TMqU3mfcqGvrk...,"( Lazy, Bankrupt, coward, Selfish)"
2335,https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjcOzY,[https://youtu.be/P3vXLJtyZyM?si=eHoU0CNW3tTjc...,
3133,https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrSbZQ,[https://youtu.be/BwbkBmplECU?si=TeRz_D0cMEOrS...,


In [11]:
df['has_mention'] = df['clean_comment'].str.contains(r'@\w+', na=False)

df[df['has_mention']][['clean_comment']].head(5)

,clean_comment
170,​ @mintetube391 😂
178,​ @Dagibeat21 ????
204,​ @bmsk0076 I hope we have Soon ! I wish .
266,@Dawit Getachew: Really በጉብዝናህ ወራት ጌታን አስበሃል፤ ...
711,​ @AdewaKedir 👌👌👌


In [12]:
df['has_hashtag'] = df['clean_comment'].str.contains(r'#\w+', na=False)

df[df['has_hashtag']][['clean_comment']].head(5)

,clean_comment
159,100% #1
580,amanu anteko erash tileyaleh uuuffff the poem ...
2640,▪️▫️ዘላለማዊ ድልና ክብር ለጀግናው #የኢትዬጵያ_የሀገር_ደህንነት_እና_...
2801,#27 አመት መስረቅ ስራ ነው እየተባለ በ #ህውሀት ሊቡች የተመራች #ኢት...
3019,#ፖርቲያችን ብልፅግና ባለ ብዙ ራዕይ እና ይቻላል ተብሎ የማይገመተውን ተ...


In [13]:
import re

def remove_tags(text):
    text = str(text)
    # Remove @mentions and #hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove extra whitespace left behind
    text = ' '.join(text.split())
    return text

df['clean_comment'] = df['clean_comment'].apply(remove_tags)
print("Mentions and Hashtags removed.")

Mentions and Hashtags removed.


In [14]:
df[df['clean_comment'].str.contains(r'[\d]+\s*[+\-*/=]\s*[\d]+', na=False)][['clean_comment']].head(5)

,clean_comment
482,አዎ እኛ ጴንጤ ኢየሱስ ክርስቶስ እርሱ ልክ እንደ አብ አምላክ ነው። ኢየ...
1923,E HPA Lost 4 minutes with out any Ground point...
1982,Fake election 0+0=0 wasting time and money
2898,ይታረም ኢትዮጵያ 1 - 0 ኬንያ
3383,ኢሳያስ ባለፈው ሳምንት የ80 -90 አመት አዛውንቶችን አና እድሜያቸው 1...


In [15]:
df[df['clean_comment'].str.contains(r'\d', na=False)][['clean_comment']].head(5)

,clean_comment
26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢
64,"80% of all Ethiopians life is like this, the r..."
80,ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...
98,4ኪሎ ዳይፐር ጨምሯል
106,ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ


In [16]:
punct_pattern = r"[.,!?;:\"'()\-]"

df[df['clean_comment'].str.contains(punct_pattern, na=False)][['clean_comment']].head(5)

,clean_comment
16,"May GOD send them courage ,Amen"
25,የሰራዊት ሁሉ ጌታ እግዚአብሔር መፅናናቱን ይስጣችሁ በኢየሱስ ክርስቶስ ታ...
26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢
28,በጣም ያማል! እግዚአብሔር ያጽናናችሁ!
31,እምባዬን መቆጣጠር አቃተኝ በጣም ያሳዝናል ። ፈጣሪ ያፅናችሁ !!


In [17]:
# Show comments containing numbers
print('Comments with numbers before cleaning:')
display(df[df['clean_comment'].str.contains(r'\d', na=False, regex=True)].head())

# Remove numbers from the clean_comment column
df['clean_comment'] = df['clean_comment'].str.replace(r'\d+', '', regex=True)

print('\nDataframe after removing numbers:')
display(df.head())


Comments with numbers before cleaning:


,index,comment,urls,clean_comment,has_mention,has_hashtag
26,26,እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢,[],እግዚኣቢሄር ነፍሳቸውን ይማር 3 3:10 3:15 😢😢😢😢,False,False
64,64,"80% of all Ethiopians life is like this, the ...",[],"80% of all Ethiopians life is like this, the r...",False,False
80,80,ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...,[],ክብርለጀግኖቻችንእንኳንለ65ኛዉየልዩኮማንዶየክብርበዓልአደረሳችሁዉጊያቀያሪየ...,False,False
98,98,4ኪሎ ዳይፐር ጨምሯል,[],4ኪሎ ዳይፐር ጨምሯል,False,False
106,106,ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ,[],ክፍል 8 ይለቀቅ በሰላም ነው ዛሬ ዘገያቹ,False,False



Dataframe after removing numbers:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False


In [18]:
# Keep ONLY Amharic characters, emojis, and whitespace
import re

def keep_amharic_and_emoji(text):
    text = str(text)
    # Regex to NOT match Amharic, Whitespace, and Emojis ranges
    # Everything else will be replaced with ''
    pattern = r'[^\u1200-\u137F\s\U00010000-\U0010FFFF\u2600-\u27BF]'
    text = re.sub(pattern, '', text)
    # Clean up extra spaces
    return ' '.join(text.split())

print('Dataframe before keeping only Amharic & Emoji:')
display(df.head())

df['clean_comment'] = df['clean_comment'].apply(keep_amharic_and_emoji)

print('\nDataframe after keeping ONLY Amharic & Emoji:')
display(df.head())

display(df.shape)

Dataframe before keeping only Amharic & Emoji:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False



Dataframe after keeping ONLY Amharic & Emoji:


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False


(55360, 6)

In [19]:
df = df[df['clean_comment'].str.strip() != '']
df = df.dropna(subset=['clean_comment'])
print(df.shape)
df.head(20)

(33136, 6)


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False
5,5,😢😢😢😢,[],😢😢😢😢,False,False
6,6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,[],የአርሲዎቹንም እገዚአብሔር ያስባቸው,False,False
7,7,ነፍስ ይማር🙏🙏,[],ነፍስ ይማር🙏🙏,False,False
8,8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,[],የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,False,False
9,9,ነፍስ የማር😢,[],ነፍስ የማር😢,False,False


In [20]:
# Comprehensive Amharic Stopwords List
amharic_stopwords = {
    "እና", "ወደ", "ከ", "ስለ", "ነው", "ነበር", "ጋር", "ደግሞ", "ብቻ", "ወዘተ", "ላይ", "ውስጥ", "በ", "እንጂ", "እንደ", "ነገር", "ግን", "ሁሉ", "አንድ", "ይህ", "ጋር", "ገና", "በኋላ",
    "ማነው", "ማን", "ምንም", "ምንድን", "ምናልባት", "እስከ", "እስከዚህ", "እስከዚያ", "እዚህ", "እዚያ", "እነዚህ", "እነዚያ", "እኔ", "አንተ", "አንቺ", "እሱ", "እሷ", "እኛ", "እናንተ", "እነሱ",
    "ናት", "ናቸው", "ነን", "ነህ", "ነሽ", "ነኝ", "ነበረ", "ነበሩ", "ነበረች", "የሆነ", "የሆኑ", "የሆነች", "ወይም", "ወይስ", "ቢሆንም", "በጣም", "አይደለም", "አይደሉም", "አይደለችም", 
    "ለ", "መሆኑን", "መሆኑ", "ማለት", "ሊሆን", "ሊሆኑ", "አሁን", "ብለዋል", "እንደነበር", "ይህም", "ብለው", "ነገሩ", "ሌሎች", "ጋር", "መሆኑን", "መሆናቸውን"
}

def remove_stopwords(text):
    return " ".join([word for word in str(text).split() if word not in amharic_stopwords])

df['clean_comment'] = df['clean_comment'].apply(remove_stopwords)
df = df[df['clean_comment'].str.strip() != '']
print(df.shape)
df.head(10)

(33117, 6)


,index,comment,urls,clean_comment,has_mention,has_hashtag
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False
5,5,😢😢😢😢,[],😢😢😢😢,False,False
6,6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,[],የአርሲዎቹንም እገዚአብሔር ያስባቸው,False,False
7,7,ነፍስ ይማር🙏🙏,[],ነፍስ ይማር🙏🙏,False,False
8,8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,[],የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,False,False
9,9,ነፍስ የማር😢,[],ነፍስ የማር😢,False,False


In [21]:
df['tokenized_comment'] = df['clean_comment'].str.split()
print(df.shape)
display(df.head())

(33117, 7)


,index,comment,urls,clean_comment,has_mention,has_hashtag,tokenized_comment
0,0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,[],እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,False,False,"[እግዚአብሔር, አምላክ, ነፍሳቸውን, በአጸደ, ገነት, ያሳርፍላቸው]"
1,1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና \nለመላዉ ኢትዮጵያ...,[],ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,False,False,"[ነፍሳችሁን, በአፀደ, ገነት, ያኑርልን, 🙏🏼ለቤተሰቦቻቸዉና, ለመላዉ, ..."
2,2,በጣም ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,[],ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,False,False,"[ያሳዝናል, ፈጣሪ, መጽናናትን, ያድላቸዉ, 🙏🙏🙏]"
3,3,😢😢😢😢😭😭😭,[],😢😢😢😢😭😭😭,False,False,[😢😢😢😢😭😭😭]
4,4,😢😢😢😢😢,[],😢😢😢😢😢,False,False,[😢😢😢😢😢]


In [22]:
!pip install scikit-learn

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize Vectorizer to accept LISTS of tokens (this preserves emojis)
vectorizer = TfidfVectorizer(analyzer='word', tokenizer=lambda x: x, preprocessor=lambda x: x, token_pattern=None)

# Use 'tokenized_comment' (the list) NOT the joined string
tfidf_matrix = vectorizer.fit_transform(df['tokenized_comment'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (33117, 86016)


In [24]:
# Get the feature names
feature_names = vectorizer.get_feature_names_out()

# Show 50 words from the middle of the list (where actual Amharic words are)
print(feature_names[1000:1050])

['ህዝብጋ' 'ህዝብጥያቄው' 'ህዝብ❤' 'ህዝብ❤❤❤❤' 'ህዝቦቹን' 'ህዝቦቹዋ' 'ህዝቦቹዋን' 'ህዝቦቻ' 'ህዝቦች'
 'ህዝቦችህ' 'ህዝቦችህን' 'ህዝቦችና' 'ህዝቦችን' 'ህዝቦችዋ' 'ህዝቦችዋን' 'ህዝቦች❤❤❤❤😢😢😢😢' 'ህዝቦቿ'
 'ህዝቦቿን' 'ህዝቦን' 'ህዝቧ' 'ህዝቧቿ' 'ህዝቧን' 'ህዝቧዞች' 'ህዝን' 'ህዝንብና' 'ህየፈለግን' 'ህያብ'
 'ህያወቱን' 'ህያው' 'ህያውነትንና' 'ህይ' 'ህይወቱ' 'ህይወቱን' 'ህይወቱንና' 'ህይወታቸውንም' 'ህይወታቸው።'
 'ህይወታችሁን' 'ህይወታችን' 'ህይወታችንን' 'ህይወቴን' 'ህይወት' 'ህይወትህ' 'ህይወትም' 'ህይወትሺን'
 'ህይወትሽን' 'ህይወትን' 'ህይወቷ' 'ህይውትህን' 'ህይዎታቸው' 'ህይዎት']


In [37]:
# Expanded lists to catch almost everything on Fana TV
pos_lexicon = {
    "ጥሩ", "አሪፍ", "ምርጥ", "ደስ", "አሜን", "በርታ", "ትክክል", "ሰላም", "እግዚአብሔር", "ፈጣሪ", 
    "ይባርክ", "ቆንጆ", "አመሰግናለሁ", "እንኳን", "ደስተኛ", "ሀገር", "ጀግና", "አሸናፊ",
    "❤️", "🙌", "🔥", "👏", "🙏", "✅", "🥰", "💪", "✨", "💖", "💎", "💯", "👑"
}

neg_lexicon = {
    "መጥፎ", "ክፉ", "ውሸት", "ሌባ", "ሞት", "ሀዘን", "ጦርነት", "አይረባም", "ሞኝ", "ያሳዝናል", 
    "ወንጀል", "ውሸታም", "ጥፋት", "ይማር", "ነፍስ", "ቀብር", "ለቅሶ", "አዝኛለሁ", "ከባድ", "አይ",
    "😭", "😢", "☹️", "💔", "😡", "👎", "🤮", "💀", "😫", "😿", "😒", "🤬", "💩"
}

# 2. Re-label function
def quick_label(text):
    text = str(text)
    pos_score = sum(1 for word in pos_lexicon if word in text)
    neg_score = sum(2 for word in neg_lexicon if word in text) # Weighted negative
    if pos_score > neg_score: return 'Positive'
    elif neg_score > pos_score: return 'Negative'
    else: return 'Neutral'

# 3. Apply it
df['sentiment'] = df['clean_comment'].apply(quick_label)

print("Labels restored!")
print(df['sentiment'].value_counts())

Labels restored!
sentiment
Neutral     21347
Positive     6334
Negative     5436
Name: count, dtype: int64


In [38]:
def deep_calculate_label(text):
    text = str(text)
    pos_score = 0
    neg_score = 0
    
    # 1. Check for Emojis and Words even if they are glued together
    for word in pos_lexicon:
        if word in text:
            pos_score += 1
            
    for word in neg_lexicon:
        if word in text:
            neg_score += 1
            
    # 2. Final Decision
    if pos_score > neg_score:
        return 'Positive'
    elif neg_score > pos_score:
        return 'Negative'
    else:
        return 'Neutral'

# Apply to the clean_comment column directly
df['sentiment'] = df['clean_comment'].apply(deep_calculate_label)

# Show Results
print("NEW ACCURATE DISTRIBUTION:")
print(df['sentiment'].value_counts())
display(df[['clean_comment', 'sentiment']].head(15))

NEW ACCURATE DISTRIBUTION:
sentiment
Neutral     22025
Positive     6529
Negative     4563
Name: count, dtype: int64


,clean_comment,sentiment
0,እግዚአብሔር አምላክ ነፍሳቸውን በአጸደ ገነት ያሳርፍላቸው,Positive
1,ነፍሳችሁን በአፀደ ገነት ያኑርልን 🙏🏼ለቤተሰቦቻቸዉና ለመላዉ ኢትዮጵያ ህ...,Positive
2,ያሳዝናል ፈጣሪ መጽናናትን ያድላቸዉ 🙏🙏🙏,Positive
3,😢😢😢😢😭😭😭,Negative
4,😢😢😢😢😢,Negative
5,😢😢😢😢,Negative
6,የአርሲዎቹንም እገዚአብሔር ያስባቸው,Neutral
7,ነፍስ ይማር🙏🙏,Negative
8,የሞቱትን ነፍሳቸውን ይማር ለቤተሠቦቻቸው መፅናናትን ይስጣቸው አሜን,Neutral
9,ነፍስ የማር😢,Negative


In [39]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [40]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Prepare the text and labels
sentences = df['clean_comment'].astype(str).tolist()
# Convert labels to numbers (0: Negative, 1: Neutral, 2: Positive)
label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
labels = df['sentiment'].map(label_map).tolist()

# 2. Tokenization (Turning words into IDs)
vocab_size = 10000 # Top 10,000 Amharic words
max_length = 100   # Max length of a comment
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(sentences)
word_index = tokenizer.word_index

# 3. Sequencing and Padding
sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# 4. Split into Training and Testing
X_train, X_test, y_train, y_test = train_test_split(padded, np.array(labels), test_size=0.2, random_state=42)

print(f"Data ready! Vocabulary size: {len(word_index)}")

Data ready! Vocabulary size: 86017


In [42]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# 1. Define the Architecture
model = Sequential([
    # Embedding: Turns word IDs into meaningful vectors
    Embedding(10000, 64, input_length=100), 
    
    # LSTM: The "Brain" that remembers the context of words
    Bidirectional(LSTM(64, return_sequences=True)),
    Bidirectional(LSTM(32)),
    
    # Dense layers for the final decision
    Dense(64, activation='relu'),
    Dropout(0.5), # This prevents the model from over-fitting
    Dense(3, activation='softmax') # Outputs probability for 3 classes: Neg, Neu, Pos
])

# 2. Compile the Model
model.compile(loss='sparse_categorical_crossentropy', 
              optimizer='adam', 
              metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [43]:
# Start the actual Deep Learning process
# We use epochs=5 to give it 5 rounds of study
history = model.fit(
    X_train, y_train, 
    epochs=5, 
    validation_data=(X_test, y_test), 
    batch_size=64, 
    verbose=1
)

print("\n--- Training Finished! ---")

Epoch 1/5
414/414 ━━━━━━━━━━━━━━━━━━━━ 92s 191ms/step - accuracy: 0.7117 - loss: 0.7446 - val_accuracy: 0.8060 - val_loss: 0.5538
Epoch 2/5
414/414 ━━━━━━━━━━━━━━━━━━━━ 76s 183ms/step - accuracy: 0.8470 - loss: 0.4525 - val_accuracy: 0.8477 - val_loss: 0.4416
Epoch 3/5
414/414 ━━━━━━━━━━━━━━━━━━━━ 80s 193ms/step - accuracy: 0.8814 - loss: 0.3478 - val_accuracy: 0.8514 - val_loss: 0.4398
Epoch 4/5
414/414 ━━━━━━━━━━━━━━━━━━━━ 76s 184ms/step - accuracy: 0.8964 - loss: 0.3008 - val_accuracy: 0.8525 - val_loss: 0.4686
Epoch 5/5
414/414 ━━━━━━━━━━━━━━━━━━━━ 76s 183ms/step - accuracy: 0.9071 - loss: 0.2680 - val_accuracy: 0.8258 - val_loss: 0.5389

--- Training Finished! ---


In [44]:
def predict_sentiment(text):
    # 1. Clean and tokenize the input
    sequence = tokenizer.texts_to_sequences([text])
    test_padded = pad_sequences(sequence, maxlen=100, padding='post')
    
    # 2. Get prediction
    prediction = model.predict(test_padded)
    classes = ['Negative', 'Neutral', 'Positive']
    
    # 3. Show result
    result = classes[np.argmax(prediction)]
    print(f"Comment: {text}")
    print(f"Predicted Sentiment: {result}")

# Try it with a test sentence!
predict_sentiment("በጣም ምርጥ ዝግጅት ነው በርቱ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Comment: በጣም ምርጥ ዝግጅት ነው በርቱ
Predicted Sentiment: Positive
